# LSTM — From Scratch (NumPy) then PyTorch

---

## What is a Neural Network?

A neural network is a chain of matrix multiplications with a non-linear function applied after each one:

- **Input** — your features (lagged returns, volatility, momentum)
- **Weights** — numbers the network learns
- **Output** — your prediction

The simplest version: `output = activation(W * input + b)`. The network learns `W` and `b` by making a prediction, measuring how wrong it was (loss), then adjusting the weights in the direction that reduces the error. That process is called **backpropagation** — it uses the chain rule to figure out how much each weight contributed to the error.

---

## Why Can't a Normal Network Handle Time Series?

A standard neural network treats every input independently — it has no memory of yesterday. For stock returns that is a problem because the sequence matters. A 3-day drop followed by a bounce is very different from a random bounce.

---

## What is an RNN?

A Recurrent Neural Network (RNN) adds a **hidden state** — a vector passed from one time step to the next. At each day it takes today's input and yesterday's hidden state and produces a new hidden state. The problem is that over long sequences, gradients during backpropagation either **vanish** (go to zero) or **explode** (go huge), so it cannot learn patterns spanning more than ~10 steps.

---

## What is an LSTM?

An LSTM (Long Short-Term Memory) fixes the vanishing gradient problem with a more sophisticated memory system. Instead of one hidden state, it has two:

- **`h`** — hidden state (short-term memory, passed to the next step)
- **`c`** — cell state (long-term memory, a conveyor belt that runs through time with only small controlled changes)

The cell state is what prevents gradients from vanishing.

---

## The 4 Gates

The LSTM controls what flows in and out of memory using 4 gates. Each gate is a matrix multiply plus an activation function, acting like a valve.

### 1. Forget Gate `f`
How much of the old cell state should I keep?

`f = sigmoid(Wf . [h_prev, x] + bf)` — f=1 keep everything, f=0 forget everything.

### 2. Input Gate `i`
How much new information should I write to memory?

`i = sigmoid(Wi . [h_prev, x] + bi)` — a valve controlling how much gets written.

### 3. Cell Gate `g` (candidate)
What new information do we want to potentially write?

`g = tanh(Wg . [h_prev, x] + bg)` — outputs between -1 and 1. Combined with `i`, you write `i * g` to memory.

### 4. Output Gate `o`
What part of the cell state should we expose as the hidden state?

`o = sigmoid(Wo . [h_prev, x] + bo)` — then `h = o * tanh(c)`

---

## Putting It Together

At each time step (each trading day):

```
f = sigmoid(Wf . [h, x] + bf)   # forget gate
i = sigmoid(Wi . [h, x] + bi)   # input gate
g = tanh(Wg . [h, x] + bg)      # candidate values
o = sigmoid(Wo . [h, x] + bo)   # output gate

c = f * c_prev + i * g           # update cell state (long-term memory)
h = o * tanh(c)                  # update hidden state (short-term memory)

prediction = Wy . h + by         # linear layer on top
```

The cell state update `c = f * c_prev + i * g` is the heart of it — you keep a fraction of old memory and add a fraction of new information. No squashing, no vanishing gradient.

---

## Why Might LSTM Beat Linear Regression?

Linear regression sees each day in isolation. LSTM sees sequences — it can learn things like after 3 consecutive down days a bounce is more likely, or high volatility preceded by a calm period is a warning sign. Whether it beats linear regression (Test R2 0.31) on this dataset is what this notebook answers.

---

## Plan
1. **NumPy LSTM** — implement the forward pass gate by gate, then BPTT, then the training loop
2. **PyTorch LSTM** — re-implement using torch.nn.LSTM on the RTX 5080, compare speed and Test R2 against the NumPy version and linear regression baseline


$$\tanh(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}}$$

$$\sigma(x) = \frac{1}{1 + e^{-x}}$$


In [23]:
import numpy as np
import yfinance as yf 


In [24]:
class LSTM:
    def __init__(self, input_size, hidden_size):
        # ✓ YOU WROTE THIS
        self.hidden_size = hidden_size
        # REVIEW: why (hidden_size, hidden_size + input_size)? because input is [h, x] concatenated
        self.Wf = np.random.randn(hidden_size, hidden_size + input_size)
        self.Wi = np.random.randn(hidden_size, hidden_size + input_size)
        self.Wg = np.random.randn(hidden_size, hidden_size + input_size)
        self.Wo = np.random.randn(hidden_size, hidden_size + input_size)
        self.Wy = np.random.randn(1, hidden_size)
        # REVIEW: biases start at zero not random — random biases can saturate gates before training starts
        self.bf = np.zeros((hidden_size, 1))
        self.bi = np.zeros((hidden_size, 1))
        self.bg = np.zeros((hidden_size, 1))
        self.bo = np.zeros((hidden_size, 1))
        self.by = np.zeros((1, 1))

    def sigmoid(self, x):
        # ✓ YOU WROTE THIS
        return 1 / (1 + np.exp(-x))

    def forward(self, x_t, h_prev, c_prev):
        # ✓ YOU WROTE THIS
        # REVIEW: h_prev first, then x_t — order must match weight matrix column order
        xh = np.concatenate([h_prev, x_t])

        f = self.sigmoid(np.dot(self.Wf, xh) + self.bf)  # forget: how much old memory to keep
        i = self.sigmoid(np.dot(self.Wi, xh) + self.bi)  # input: how much new info to write
        g = np.tanh(np.dot(self.Wg, xh) + self.bg)       # candidate: what new info to write
        o = self.sigmoid(np.dot(self.Wo, xh) + self.bo)  # output: what to expose as hidden state

        c = f * c_prev + i * g   # keep fraction of old memory, add fraction of new info
        h = o * np.tanh(c)       # filtered view of cell state

        prediction = np.dot(self.Wy, h) + self.by
        # REVIEW: return f,i,g,o so backward() can use them for derivatives
        return h, c, f, i, g, o, prediction

    def backward(self, x_t, h_prev, c_prev, h, c, f, i, g, o, prediction, y_true):
        # ✓ YOU WROTE THIS SECTION
        dL_dpred = 2 * (prediction - y_true)

        dWy = np.dot(dL_dpred, h.T)
        dby = dL_dpred
        dh  = np.dot(self.Wy.T, dL_dpred)

        do = dh * np.tanh(c)
        dc = dh * o * (1 - np.tanh(c) ** 2)  # (1 - tanh^2) is the tanh derivative

        df = dc * c_prev
        di = dc * g
        dg = dc * i

        # ── CLAUDE WROTE FROM HERE ──────────────────────────────────────────
        # REVIEW: chain rule back through the activation functions
        df_raw = df * f * (1 - f)   # sigmoid derivative: s * (1 - s)
        di_raw = di * i * (1 - i)   # sigmoid derivative
        dg_raw = dg * (1 - g ** 2)  # tanh derivative: 1 - tanh^2
        do_raw = do * o * (1 - o)   # sigmoid derivative

        # REVIEW: weight gradients = gate_delta . input^T (outer product)
        xh = np.concatenate([h_prev, x_t])
        dWf = np.dot(df_raw, xh.T)
        dWi = np.dot(di_raw, xh.T)
        dWg = np.dot(dg_raw, xh.T)
        dWo = np.dot(do_raw, xh.T)
        dbf = df_raw
        dbi = di_raw
        dbg = dg_raw
        dbo = do_raw

        return dWf, dWi, dWg, dWo, dWy, dbf, dbi, dbg, dbo, dby

    def train(self, X, y, epochs=100, learning_rate=0.001):
        # ── CLAUDE WROTE THIS ───────────────────────────────────────────────
        # REVIEW: reset h and c to zero each epoch — fresh memory each pass through data
        for epoch in range(epochs):
            h = np.zeros((self.hidden_size, 1))
            c = np.zeros((self.hidden_size, 1))
            total_loss = 0

            for t in range(len(X)):
                x_t    = X[t].reshape(-1, 1)
                y_true = y[t].reshape(1, 1)
                h_prev = h
                c_prev = c

                # h and c carry memory forward to the next time step
                h, c, f, i, g, o, pred = self.forward(x_t, h_prev, c_prev)
                total_loss += np.mean((pred - y_true) ** 2)

                grads = self.backward(x_t, h_prev, c_prev, h, c, f, i, g, o, pred, y_true)
                dWf, dWi, dWg, dWo, dWy, dbf, dbi, dbg, dbo, dby = grads

                # REVIEW: gradient descent — nudge each weight opposite to its gradient
                self.Wf -= learning_rate * dWf
                self.Wi -= learning_rate * dWi
                self.Wg -= learning_rate * dWg
                self.Wo -= learning_rate * dWo
                self.Wy -= learning_rate * dWy
                self.bf -= learning_rate * dbf
                self.bi -= learning_rate * dbi
                self.bg -= learning_rate * dbg
                self.bo -= learning_rate * dbo
                self.by -= learning_rate * dby

            if epoch % 10 == 0:
                print(f'Epoch {epoch:3d}  Loss: {total_loss / len(X):.6f}')

In [25]:
import torch
import torch.nn as nn
import pandas as pd

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Using device: {device}')

class LSTMPyTorch(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.lstm    = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(0.2)
        self.linear  = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        last_h = self.dropout(out[:, -1, :])
        return self.linear(last_h)

Using device: mps


In [26]:
feature_cols = ['Lag1', 'Lag2', 'Lag3', 'Vol_5day', 'Vol_30day',
                'Momentum_5days', 'Momentum_30days', 'Momentum_252days']

def make_sequences(ticker, seq_len=20):
    data = yf.Ticker(ticker).history(period='5y')
    data['Return'] = data['Close'].pct_change()
    data['Lag1'] = data['Return'].shift(1)
    data['Lag2'] = data['Return'].shift(2)
    data['Lag3'] = data['Return'].shift(3)
    data['Vol_5day'] = data['Return'].rolling(5).std()
    data['Vol_30day'] = data['Return'].rolling(30).std()
    data['Momentum_5days'] = data['Return'].rolling(5).mean()
    data['Momentum_30days'] = data['Return'].rolling(30).mean()
    data['Momentum_252days'] = data['Return'].rolling(252).mean()
    data.dropna(inplace=True)

    X = data[feature_cols].values
    y = data['Return'].values

    # normalize per ticker
    X = (X - X.mean(axis=0)) / X.std(axis=0)

    # build sequences within this ticker only
    X_seq, y_seq = [], []
    for i in range(seq_len, len(X)):
        X_seq.append(X[i-seq_len:i])
        y_seq.append(y[i])

    return np.array(X_seq), np.array(y_seq)

tickers = ['AAPL', 'MSFT', 'GOOGL', 'SPY', 'AMZN', 'META', 'NVDA', 'TSLA', 'JPM', 'NFLX']
all_X, all_y = [], []
for ticker in tickers:
    X_seq, y_seq = make_sequences(ticker)
    all_X.append(X_seq)
    all_y.append(y_seq)
    print(f'{ticker}: {len(X_seq)} sequences')

X_all = np.concatenate(all_X)
y_all = np.concatenate(all_y)

split = int(len(X_all) * 0.8)
X_train = torch.tensor(X_all[:split], dtype=torch.float32).to(device)
X_test  = torch.tensor(X_all[split:], dtype=torch.float32).to(device)
y_train = torch.tensor(y_all[:split], dtype=torch.float32).to(device)
y_test  = torch.tensor(y_all[split:], dtype=torch.float32).to(device)

print(f'\nX_train: {X_train.shape}  X_test: {X_test.shape}')

AAPL: 983 sequences
MSFT: 983 sequences
GOOGL: 983 sequences
SPY: 983 sequences
AMZN: 983 sequences
META: 983 sequences
NVDA: 983 sequences
TSLA: 983 sequences
JPM: 983 sequences
NFLX: 983 sequences

X_train: torch.Size([7864, 20, 8])  X_test: torch.Size([1966, 20, 8])


In [27]:
model = LSTMPyTorch(input_size=8, hidden_size=32).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.MSELoss()

epochs = 200
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    pred = model(X_train).squeeze()
    loss = loss_fn(pred, y_train)
    loss.backward()
    optimizer.step()

    if epoch % 20 == 0:
        model.eval()
        with torch.no_grad():
            test_pred = model(X_test).squeeze()
            test_loss = loss_fn(test_pred, y_test)
        print(f'Epoch {epoch:3d}  Train Loss: {loss.item():.6f}  Test Loss: {test_loss.item():.6f}')

Epoch   0  Train Loss: 0.004467  Test Loss: 0.002271
Epoch  20  Train Loss: 0.001488  Test Loss: 0.000503
Epoch  40  Train Loss: 0.001111  Test Loss: 0.000437
Epoch  60  Train Loss: 0.000912  Test Loss: 0.000422
Epoch  80  Train Loss: 0.000795  Test Loss: 0.000414
Epoch 100  Train Loss: 0.000738  Test Loss: 0.000411
Epoch 120  Train Loss: 0.000695  Test Loss: 0.000411
Epoch 140  Train Loss: 0.000676  Test Loss: 0.000410
Epoch 160  Train Loss: 0.000648  Test Loss: 0.000409
Epoch 180  Train Loss: 0.000627  Test Loss: 0.000409


In [28]:
model.eval()
with torch.no_grad():
    y_pred_train = model(X_train).squeeze().cpu().numpy()
    y_pred_test  = model(X_test).squeeze().cpu().numpy()

y_train_np = y_train.cpu().numpy()
y_test_np  = y_test.cpu().numpy()

mse = np.mean((y_test_np - y_pred_test) ** 2)

ss_res = np.sum((y_test_np - y_pred_test) ** 2)
ss_tot = np.sum((y_test_np - np.mean(y_test_np)) ** 2)
r2 = 1 - ss_res / ss_tot

ss_res_train = np.sum((y_train_np - y_pred_train) ** 2)
ss_tot_train = np.sum((y_train_np - np.mean(y_train_np)) ** 2)
r2_train = 1 - ss_res_train / ss_tot_train

print(f'MSE:      {mse:.6f}')
print(f'Train R²: {r2_train:.4f}')
print(f'Test R²:  {r2:.4f}')

MSE:      0.000408
Train R²: 0.0143
Test R²:  -0.0077
